In [1]:
#@title 0.1 Clone Repository & Install Dependencies
import os

# --- Set the data directory (adjust if running locally) ---
DATA_DIR = '/content/LLM_IP_assignment'   # <-- Colab default
# DATA_DIR = '.'   # <-- uncomment if running locally inside the repo

if not os.path.exists(DATA_DIR):
    !git clone https://github.com/matthewdelorenzo/LLM_IP_assignment.git {DATA_DIR}

os.chdir(DATA_DIR)
print('Working directory:', os.getcwd())

# Install Python dependencies
!pip install -q networkx openai

# Install Yosys (formal equivalence checker)
!apt-get install -y -q yosys

# Make the SIM binary executable
SIM_EXECUTABLE = os.path.join(DATA_DIR, 'sim', 'sim_text')
if os.path.exists(SIM_EXECUTABLE):
    os.chmod(SIM_EXECUTABLE, 0o755)
    print('SIM binary ready.')
else:
    print('WARNING: SIM binary not found at', SIM_EXECUTABLE)

Cloning into '/content/LLM_IP_assignment'...
remote: Enumerating objects: 487, done.
remote: Counting objects: 100% (487/487), done.
remote: Compressing objects: 100% (267/267), done.
remote: Total 487 (delta 125), reused 482 (delta 122), pack-reused 0 (from 0)
Receiving objects: 100% (487/487), 4.26 MiB | 12.27 MiB/s, done.
Resolving deltas: 100% (125/125), done.
Working directory: /content/LLM_IP_assignment
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core berkeley-abc gir1.2-atk-1.0 gir1.2-gtk-3.0
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common librsvg2-common
  libxcomposite1 libxtst6 python3-cairo python3-gi-cairo python3-numpy
  session-migration xdot
Suggested packages:
  gvfs python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  at-spi2-core berkeley-abc gir1.2-

In [2]:
#@title 0.2 Imports and Helper Functions
import sys, os, glob, pickle, copy, random, subprocess, time, re
import networkx as nx
import openai

# Add src/ directories to path
sys.path.insert(0, os.path.join(DATA_DIR, 'src'))
sys.path.insert(0, os.path.join(DATA_DIR, 'src_ablation_study_wo_SolB'))

# Core functions from original codebase
from characterize_circuit import (
    characterize_generic_netlist,
    create_LLM_circuit_query_template,
    is_gate_line,
    convert_circuit_to_Boolean_format,
    convert_circuit_from_Boolean_format,
)
from circuit_decomposer import (
    parse_expression,
    decompose_expression,
    final_decompose_expression,
)
from sim_evaluation import evaluate_sim_text
from config import DATASET_PATH, SIM_DIR

# Helper functions for the workshop
def evaluate_sim(orig_contents, pirated_contents, label='test'):
    """Save circuits to temp files and run SIM text comparison."""
    os.makedirs('tmp_eval', exist_ok=True)
    orig_path = f'tmp_eval/orig_{label}.v'
    pirated_path = f'tmp_eval/pirated_{label}.v'

    with open(orig_path, 'w') as f:
        f.write(orig_contents)
    with open(pirated_path, 'w') as f:
        f.write(pirated_contents)

    try:
        result = evaluate_sim_text(orig_path, pirated_path)
        if result.returncode != 0:
            print(f'SIM error (exit code {result.returncode}): {result.stderr}')
            return None
        if 'consists for' in result.stdout:
            score = float(
                result.stdout.split('\n')[-2]
                .split('consists for')[-1]
                .split('%')[0].strip()
            ) / 100.0
            return score
        return 0.0
    except Exception as e:
        print(f'SIM error: {e}')
        return None


def _strip_code_fences(text):
    """Remove markdown code fences from LLM responses."""
    if '```' in text:
        parts = text.split('```')
        if len(parts) >= 2:
            content = parts[1]
            lines = content.split('\n')
            if lines and lines[0].strip().lower() in ('', 'verilog', 'v', 'sv', 'systemverilog'):
                content = '\n'.join(lines[1:])
            return content.strip()
    return text


def _fix_verilog_declarations(verilog):
    """
    Fix common LLM Verilog output issues:
    - Add missing semicolons to input/output/wire/reg declaration lines
      that end a statement but are missing the terminating ';'
    """
    _DECL_KW = re.compile(r'^\s*(input|output|inout|wire|reg)\b')
    fixed_lines = []
    lines = verilog.splitlines()
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        if _DECL_KW.match(stripped):
            # Only add ';' if the line doesn't already end with ';' or ','
            # and it's not a continuation line (next line also starts a declaration
            # or is blank / endmodule)
            if stripped and not stripped.endswith(';') and not stripped.endswith(','):
                stripped = stripped + ';'
        fixed_lines.append(stripped)
    return '\n'.join(fixed_lines)


def _get_module_name(verilog):
    """Extract the top module name from a Verilog string."""
    for line in verilog.splitlines():
        m = re.match(r'\s*module\s+(\w+)', line)
        if m:
            return m.group(1)
    return None


def check_functional_equivalence(orig_verilog, transformed_verilog, label='test'):
    """
    Formal equivalence check using Yosys miter circuit + SAT solver.

    Runs the following Yosys flow:
        read_verilog
        read_verilog
        prep; proc; opt; memory;
        clk2fflogic;
        miter -equiv -flatten   miter
        sat -seq 50 -verify -prove trigger 0 -show-all \
            -show-inputs -show-outputs -set-init-zero miter

    Returns: (is_equivalent: bool | None, details: str)
        True  — formally proven equivalent
        False — counterexample found (not equivalent)
        None  — Yosys error or inconclusive result
    """
    os.makedirs('tmp_eval', exist_ok=True)

    # Strip markdown fences and fix common LLM Verilog formatting issues
    orig_clean = _fix_verilog_declarations(_strip_code_fences(orig_verilog))
    trans_clean = _fix_verilog_declarations(_strip_code_fences(transformed_verilog))

    orig_module = _get_module_name(orig_clean)
    gen_module  = _get_module_name(trans_clean)

    if not orig_module:
        return None, "Could not extract module name from original Verilog"
    if not gen_module:
        return None, "Could not extract module name from transformed Verilog"

    # Rename both modules to unique names so Yosys never sees a name clash
    gold_name = 'gold_circuit'
    gate_name = 'gate_circuit'

    gold_verilog = re.sub(
        r'\bmodule\s+' + re.escape(orig_module) + r'\b',
        f'module {gold_name}', orig_clean, count=1
    )
    gate_verilog = re.sub(
        r'\bmodule\s+' + re.escape(gen_module) + r'\b',
        f'module {gate_name}', trans_clean, count=1
    )

    truth_path  = os.path.abspath(f'tmp_eval/truth_{label}.v')
    gen_path    = os.path.abspath(f'tmp_eval/gen_{label}.v')
    script_path = os.path.abspath(f'tmp_eval/eq_{label}.ys')

    with open(truth_path, 'w') as f:
        f.write(gold_verilog)
    with open(gen_path, 'w') as f:
        f.write(gate_verilog)

    yosys_script = (
        f"read_verilog {truth_path}\n"
        f"read_verilog {gen_path}\n"
        "prep; proc; opt; memory;\n"
        "clk2fflogic;\n"
        f"miter -equiv -flatten {gate_name} {gold_name} miter\n"
        "sat -seq 50 -verify -prove trigger 0 "
        "-show-all -show-inputs -show-outputs -set-init-zero miter\n"
    )
    with open(script_path, 'w') as f:
        f.write(yosys_script)

    print(f"🔍 Running Yosys equivalence check for '{label}'...")
    try:
        result = subprocess.run(
            ['yosys', '-s', script_path],
            capture_output=True, text=True, timeout=180
        )
        output = result.stdout + result.stderr

        if 'no model found: SUCCESS' in output:
            return True, "Circuits formally verified equivalent (Yosys miter+SAT)"
        elif 'proof did fail' in output:
            return False, "Circuits NOT equivalent — Yosys SAT found a counterexample"
        elif result.returncode != 0:
            err_lines = [l.strip() for l in output.splitlines() if 'ERROR' in l or 'Error' in l]
            err_msg = '; '.join(err_lines[:3]) if err_lines else output[-400:]
            return None, f"Yosys error: {err_msg}"
        else:
            return None, f"Yosys result inconclusive (exit {result.returncode})"

    except subprocess.TimeoutExpired:
        return None, "Yosys equivalence check timed out (180 s)"
    except FileNotFoundError:
        return None, "Yosys not found — run: !apt-get install -y yosys  (or re-run cell 3)"
    except Exception as e:
        return None, f"Equivalence check error: {e}"


def get_mapped_circuit(orig_file_contents, cached_circuit_mapping, mapping_strategy, rank=0):
    """
    Replace every gate in the original circuit with the LLM's cached rephrase.
    From evaluate_piracy_using_cached_mapping_multiprocessing_v2.py in the original codebase.
    """
    new_lines = []
    new_wires = []
    lines = orig_file_contents.split(';')
    lines = [line.strip() for line in lines if line.strip()]
    lines = [line.replace('\n', ' ') for line in lines]
    for line in lines:
        if line.startswith(('module ', 'input ', 'output ', 'wire ', 'endmodule')):
            new_lines.append(line)
            continue
        if is_gate_line(line):
            keep_line_same = False
            gate_type = line.split()[0] + '_' + str(line.count(','))
            if gate_type not in cached_circuit_mapping or len(cached_circuit_mapping[gate_type]) == 0:
                new_lines.append(line)
                continue
            if len(list(cached_circuit_mapping[gate_type].keys())) > 0:
                line = line.replace('(', ' ( ')
                line = line.replace(')', ' ) ')
                gate_type = line.split(' ')[0] + '_' + str(line.count(','))
                gate_name = line.split()[1].strip()
                op = line.split()[3].strip(',').strip()
                num_ips = line.count(',')
                ips = [line.split()[4+j].strip().strip(',').strip() for j in range(num_ips)]
                if mapping_strategy == 'random':
                    target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                else:
                    if mapping_strategy.split('_')[0] == gate_type.split('_')[0].upper():
                        keep_line_same = True
                    elif mapping_strategy in list(cached_circuit_mapping[gate_type].keys()):
                        target_mapping_gate = copy.deepcopy(mapping_strategy)
                    else:
                        target_mapping_gate = random.choice(list(cached_circuit_mapping[gate_type].keys()))
                if keep_line_same:
                    new_lines.append(line)
                else:
                    mapped_circuit_str = cached_circuit_mapping[gate_type][target_mapping_gate][0]
                    nets = []
                    mapped_lines = mapped_circuit_str.split('\n')
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        nets.extend(tmp_inputs)
                        nets.append(tmp_output)
                    nets = list(set(nets))
                    inter_net_cnt = 0
                    for net in nets:
                        if net == 'Y':
                            for pat in ['Y = ', 'Y= ', 'Y =', 'Y=']:
                                if pat in mapped_circuit_str:
                                    mapped_circuit_str = mapped_circuit_str.replace(pat, op + pat[1:])
                                    break
                        elif net in ['A' + str(i) for i in range(1, 50)]:
                            idx = int(net[1:]) - 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                repl = pat.replace(net, ips[idx])
                                mapped_circuit_str = mapped_circuit_str.replace(pat, repl)
                        else:
                            wire_name = gate_name + '_inter_net_' + str(inter_net_cnt)
                            inter_net_cnt += 1
                            for pat in ['(' + net + ',', ' ' + net + ',', ' ' + net + ')', '(' + net + ')']:
                                mapped_circuit_str = mapped_circuit_str.replace(pat, pat.replace(net, wire_name))
                            if net + ' = ' in mapped_circuit_str:
                                mapped_circuit_str = mapped_circuit_str.replace(net + ' = ', wire_name + ' = ')
                            new_wires.append(wire_name)
                    mapped_lines = mapped_circuit_str.split('\n')
                    cnt = 0
                    for l in mapped_lines:
                        l = l.strip().strip(';')
                        if not l:
                            continue
                        tmp_inputs = l.split('(')[-1].split(')')[0].split(',')
                        tmp_inputs = [inp.strip() for inp in tmp_inputs]
                        tmp_output = l.split('=')[0].strip()
                        tmp_gate_type = l.split('=')[-1].split('(')[0].strip()
                        if tmp_gate_type in ['AND', 'OR', 'NAND', 'NOR'] and len(tmp_inputs) == 1:
                            tmp_inputs = [tmp_inputs[0], tmp_inputs[0]]
                        new_str = tmp_gate_type.lower() + ' ' + gate_name + '_' + str(cnt) + ' ( ' + tmp_output + ', ' + ', '.join(tmp_inputs) + ' )'
                        cnt += 1
                        new_lines.append(new_str)
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)
    if new_wires:
        for i in range(len(new_lines)):
            if new_lines[i].split()[0] == 'wire':
                new_lines[i] = new_lines[i] + ';\nwire ' + ', '.join(new_wires)
                break
    return ';\n'.join(new_lines)


def get_circuit_from_response(response):
    """Parse LLM response to extract gate equations (from original code)."""
    LLM_circuit = ""
    if "```" in response:
        lines = response.split("```")[1].split("\n")
        lines = [l for l in lines if l != '']
    elif "=" in response:
        lines = response.split("\n")
    else:
        return "Incorrect format"

    for line in lines:
        if ("=" in line) and (not line.startswith("wire ")):
            tmp_expression = line.split("=")[-1].strip().strip(";")
            if any(f"{g} " in tmp_expression for g in ["AND", "OR", "NAND", "NOR", "XOR", "XNOR"]):
                return "Incorrect format"
            tmp_op_net = line.split("=")[0].strip()
            decomposed = final_decompose_expression(tmp_expression, tmp_op_net)
            LLM_circuit += decomposed + "\n"

    if not LLM_circuit.strip():
        return "Incorrect format"
    return LLM_circuit.rstrip("\n")


SIM_THRESHOLD = 0.3

print('✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).')
print(f'SIM detection threshold: {SIM_THRESHOLD}')

✅ All imports successful. Using Yosys formal equivalence checking (miter + SAT).
SIM detection threshold: 0.3


In [ ]:
#@title 0.3 Load Circuit & Setup OpenAI Client

# Load the test circuit
DESIGN = 'C432'
VARIANT = 'c432-CS320'

circuit_path = os.path.join(DATASET_PATH, DESIGN, VARIANT, 'topModule.v')
with open(circuit_path, 'r') as f:
    orig_file_contents = f.read()

gate_counts = characterize_generic_netlist(orig_file_contents)
total_gates = sum(gate_counts.values())

print(f'Loaded circuit: {DESIGN}/{VARIANT}')
print(f'Total gates: {total_gates}')
print(f'Gate types: {dict(gate_counts)}')
print(f'Circuit size: {len(orig_file_contents)} characters')

# Show first 400 chars of the circuit
print(f'\nFirst 400 characters of circuit:')
print(orig_file_contents[:400] + '...')

# Set up OpenAI client for LLM experiments
# TODO: Set your OpenAI API key here
OPENAI_API_KEY = ""   # <-- Students: paste your API key

if not OPENAI_API_KEY:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if OPENAI_API_KEY:
    openai.api_key = OPENAI_API_KEY
    print("\n✅ OpenAI API configured")
else:
    print("\n⚠️  WARNING: No API key set. You'll need this for live LLM querying.")
    print("Set OPENAI_API_KEY above or use the environment variable.")

Loaded circuit: C432/c432-CS320
Total gates: 212
Gate types: {'nand_2': 64, 'not_1': 60, 'xor_2': 32, 'nor_2': 19, 'xnor_2': 18, 'nand_4': 14, 'and_9': 3, 'and_8': 1, 'nand_3': 1}
Circuit size: 10789 characters

First 400 characters of circuit:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
        keyIn_0_10,...

✅ OpenAI API configured


In [7]:
#@title Task 1: Direct Verilog Rewriting Implementation

# TODO: Students can modify these settings
TASK1_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "gpt-3.5-turbo-16k"  # Try: "gpt-4"

# TODO: Students should improve this prompt template
task1_prompt = f"""
Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit using ONLY primitive NOR gates.

Requirements:
1. Preserve the exact same module name, inputs, outputs, and Boolean functionality.
2. Output only valid Verilog code.
3. Do not include any explanation, comments, or markdown fences.
4. Use only primitive nor gates and wire declarations.
5. Do not optimize or simplify the logic.
6. Keep the circuit combinational unless the original circuit is sequential.
7. Implement inversion using nor(a,a).
8. Declare all intermediate wires before use.
9. Ensure the module header is valid Verilog syntax.
10. Ensure the code is complete and ends with endmodule.

Original circuit:
{orig_file_contents}
"""

print("Task 1 Prompt Summary:")
print("=" * 50)
print(f"Target gates: {TASK1_TARGET_GATES}")
print(f"Model: {TASK1_MODEL}")
print(f"Prompt length: {len(task1_prompt)} characters")
print("\nFirst 400 chars of prompt:")
print(task1_prompt[:400] + "...")

# Initialize result variables
task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

# Task 1 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 1...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        task1_result = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        # Evaluate functional correctness
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task1_result, "task1"
        )
        task1_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
        task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 1 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task1_sim_score:.4f}" if task1_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
        print(f"Model Used: {TASK1_MODEL}")
        print(f"Target Gates: {TASK1_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task1_functionally_correct and task1_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None
else:
    print("⚠️ Skipping Task 1 execution (no API key)")

Task 1 Prompt Summary:
Target gates: NOR
Model: gpt-3.5-turbo-16k
Prompt length: 11495 characters

First 400 chars of prompt:

Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit using ONLY primitive NOR gates.

Requirements:
1. Preserve the exact same module name, inputs, outputs, and Boolean functionality.
2. Output only valid Verilog code.
3. Do not include any explanation, comments, or markdown fences.
4. Use only primitive nor gates and wire declarations.
5. Do not optimize or simpli...

EXECUTING TASK 1...
✅ LLM Response received (8477 chars)
First 400 chars of response:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
  

In [9]:
#@title Task 1: Direct Verilog Rewriting Implementation

# TODO: Students can modify these settings
TASK1_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK1_MODEL = "gpt-4"

# TODO: Students should improve this prompt template
task1_prompt = f"""
Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit using ONLY primitive NOR gates.

Requirements:
1. Preserve the exact same module name, inputs, outputs, and Boolean functionality.
2. Output only valid Verilog code.
3. Do not include any explanation, comments, or markdown fences.
4. Use only primitive nor gates and wire declarations.
5. Do not optimize or simplify the logic.
6. Keep the circuit combinational unless the original circuit is sequential.
7. Implement inversion using nor(a,a).
8. Declare all intermediate wires before use.
9. Ensure the module header is valid Verilog syntax.
10. Ensure the code is complete and ends with endmodule.

Original circuit:
{orig_file_contents}
"""

print("Task 1 Prompt Summary:")
print("=" * 50)
print(f"Target gates: {TASK1_TARGET_GATES}")
print(f"Model: {TASK1_MODEL}")
print(f"Prompt length: {len(task1_prompt)} characters")
print("\nFirst 400 chars of prompt:")
print(task1_prompt[:400] + "...")

# Initialize result variables
task1_result = None
task1_sim_score = None
task1_evades = False
task1_functionally_correct = None

# Task 1 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 1...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK1_MODEL,
            messages=[{"role": "user", "content": task1_prompt}],
            temperature=0.3,
            max_tokens=2500
        )
        task1_result = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(task1_result)} chars)")
        print("First 400 chars of response:")
        print(task1_result[:400] + "...")

        # Evaluate functional correctness
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task1_result, "task1"
        )
        task1_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task1_sim_score = evaluate_sim(orig_file_contents, task1_result, "task1")
        task1_evades = task1_sim_score is not None and task1_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 1 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task1_functionally_correct else '❌ NO' if task1_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task1_sim_score:.4f}" if task1_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task1_evades else '❌ NO'}")
        print(f"Model Used: {TASK1_MODEL}")
        print(f"Target Gates: {TASK1_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task1_functionally_correct and task1_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 1 failed: {e}")
        task1_result = None
        task1_sim_score = None
        task1_evades = False
        task1_functionally_correct = None
else:
    print("⚠️ Skipping Task 1 execution (no API key)")

Task 1 Prompt Summary:
Target gates: NOR
Model: gpt-4
Prompt length: 11495 characters

First 400 chars of prompt:

Rewrite the following Verilog circuit into a functionally equivalent Verilog circuit using ONLY primitive NOR gates.

Requirements:
1. Preserve the exact same module name, inputs, outputs, and Boolean functionality.
2. Output only valid Verilog code.
3. Do not include any explanation, comments, or markdown fences.
4. Use only primitive nor gates and wire declarations.
5. Do not optimize or simpli...

EXECUTING TASK 1...
✅ LLM Response received (5403 chars)
First 400 chars of response:
module top (N1,N4,N8,N11,N14,N17,N21,N24,N27,N30,
             N34,N37,N40,N43,N47,N50,N53,N56,N60,N63,
             N66,N69,N73,N76,N79,N82,N86,N89,N92,N95,
             N99,N102,N105,N108,N112,N115,N223,N329,N370,N421,
                  N430,N431,N432,
        keyIn_0_0, keyIn_0_1, keyIn_0_2, keyIn_0_3, keyIn_0_4,
        keyIn_0_5, keyIn_0_6, keyIn_0_7, keyIn_0_8, keyIn_0_9,
        keyIn_

**Task 1 Observations**

Model used:
GPT-3.5-turbo-16k and GPT-4


SIM Score:
0.2200 for both models

Success/Failure:
Failure overall for both models. Although both models produced NOR-only rewrites that evaded the SIM-based piracy detector, neither output passed functional verification. Yosys reported a syntax error (“unexpected end of file”) in both cases, so functional correctness remained unknown.

Key insights:
Both GPT-3.5 and GPT-4 were able to generate structurally different outputs with low SIM scores, showing that direct rewriting can help evade similarity-based detection. However, both models struggled to generate complete and syntactically valid Verilog for a large NOR-only full-circuit rewrite. This suggests that for Task 1, the main challenge was output completeness and syntax correctness rather than SIM evasion.


---


**Questions：**

1. Success Rate

  No. The rewrite was not fully successful. Both GPT-3.5 and GPT-4 generated NOR-only Verilog, but Yosys reported a syntax error (“unexpected end of file”), so the circuits could not be fully verified.

2. SIM Evasion

  The SIM score was 0.2200 for both models. Since this is below 0.3, both outputs evaded detection.

3. Functional Correctness

  I would verify correctness by simulating the original and rewritten circuits with the same testbench and comparing their outputs. Formal equivalence checking with Yosys is another way. In this case, correctness was unknown because the generated Verilog had a syntax error.

4. Prompt Engineering

  I would improve the prompt by making it shorter and emphasizing complete, valid Verilog output. I would also clearly require a correct module header, declared wires, and a final endmodule.

5. Model Comparison

  GPT-3.5 and GPT-4 performed similarly here. Both got the same SIM score and both failed functional verification because of the same syntax issue. So neither model clearly performed better on this task.

In [10]:

#@title Task 2: Boolean Format Rewriting Implementation

# Convert circuit to Boolean format (this makes it easier for LLM)
circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

print("Boolean Format Conversion:")
print("=" * 30)
print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
print("\nFirst 300 chars of Boolean format:")
print(circ_in_Boolean_format[:300] + "...")

# TODO: Students configure task 2
TASK2_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "gpt-3.5-turbo-16k"

# TODO: Students should improve this prompt
task2_prompt = f"""
Rewrite the following Boolean equations into functionally equivalent Boolean equations
using ONLY the target gate set: {TASK2_TARGET_GATES}.

Strict requirements:
1. Preserve the exact same functionality.
2. Keep all original input and output variable names unchanged.
3. You may introduce intermediate variables if needed.
4. Output ONLY Boolean equation lines. Do not include explanations.
5. Do NOT use markdown code fences.
6. Each line must follow this format exactly:
   variable = GATE(input1, input2)
   or for inversion if needed:
   variable = GATE(input1)
7. Use ONLY these gate/operator names allowed by the target gate set.
8. Do not use XOR, XNOR, OR, AND, NOT, BUF, or any other operators unless they are explicitly allowed.
9. Reuse existing variable names when appropriate and ensure the equations are self-consistent.

Target gate set: {TASK2_TARGET_GATES}

Original Boolean equations:
{circ_in_Boolean_format}
"""

print(f"\nTask 2 Configuration:")
print(f"Target gates: {TASK2_TARGET_GATES}")
print(f"Model: {TASK2_MODEL}")
print(f"Prompt length: {len(task2_prompt)} characters")

# Initialize results
task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

# Task 2 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.3,
            max_tokens=4000
        )
        llm_boolean_response = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task2_sim_score:.4f}" if task2_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None
else:
    print("⚠️ Skipping Task 2 execution (no API key)")


Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NOR
Model: gpt-3.5-turbo-16k
Prompt length: 6942 characters

EXECUTING TASK 2...
✅ LLM Response received (28 chars)
First 300 chars of Boolean response:
N435 = NOR(N430, N431, N432)...

🔄 Converting back to Verilog format...
✅ Converted back to Verilog (2258 chars)

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'task2'...
Functional check: Yosys error: /content/LLM_IP_assignment/tmp_eval/gen_task2.v:2: ERROR: syntax error, unexpected TOK_INPUT, expecting ';'

🔍 Evaluating with SIM detector...

📊 

In [11]:

#@title Task 2: Boolean Format Rewriting Implementation

# Convert circuit to Boolean format (this makes it easier for LLM)
circ_in_Boolean_format, remaining_orig_lines = convert_circuit_to_Boolean_format(orig_file_contents)

print("Boolean Format Conversion:")
print("=" * 30)
print(f"Boolean equations: {len([l for l in circ_in_Boolean_format.splitlines() if l.strip()])} lines")
print(f"Remaining structural lines: {len(remaining_orig_lines.splitlines())} lines")
print("\nFirst 300 chars of Boolean format:")
print(circ_in_Boolean_format[:300] + "...")

# TODO: Students configure task 2
TASK2_TARGET_GATES = "NOR"  # Try: "NOR", "NAND", "AND and NOT"
TASK2_MODEL = "gpt-4"

# TODO: Students should improve this prompt
task2_prompt = f"""
Rewrite the following Boolean equations into functionally equivalent Boolean equations
using ONLY the target gate set: {TASK2_TARGET_GATES}.

Strict requirements:
1. Preserve the exact same functionality.
2. Keep all original input and output variable names unchanged.
3. You may introduce intermediate variables if needed.
4. Output ONLY Boolean equation lines. Do not include explanations.
5. Do NOT use markdown code fences.
6. Each line must follow this format exactly:
   variable = GATE(input1, input2)
   or for inversion if needed:
   variable = GATE(input1)
7. Use ONLY these gate/operator names allowed by the target gate set.
8. Do not use XOR, XNOR, OR, AND, NOT, BUF, or any other operators unless they are explicitly allowed.
9. Reuse existing variable names when appropriate and ensure the equations are self-consistent.

Target gate set: {TASK2_TARGET_GATES}

Original Boolean equations:
{circ_in_Boolean_format}
"""

print(f"\nTask 2 Configuration:")
print(f"Target gates: {TASK2_TARGET_GATES}")
print(f"Model: {TASK2_MODEL}")
print(f"Prompt length: {len(task2_prompt)} characters")

# Initialize results
task2_result = None
task2_sim_score = None
task2_evades = False
task2_functionally_correct = None

# Task 2 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 2...")
    print("=" * 50)

    try:
        response = openai.chat.completions.create(
            model=TASK2_MODEL,
            messages=[{"role": "user", "content": task2_prompt}],
            temperature=0.3,
            max_tokens=2500
        )
        llm_boolean_response = response.choices[0].message.content

        print(f"✅ LLM Response received ({len(llm_boolean_response)} chars)")
        print("First 300 chars of Boolean response:")
        print(llm_boolean_response[:300] + "...")

        # Convert back to Verilog format
        print("\n🔄 Converting back to Verilog format...")
        task2_result = convert_circuit_from_Boolean_format(llm_boolean_response, remaining_orig_lines)

        print(f"✅ Converted back to Verilog ({len(task2_result)} chars)")

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task2_result, "task2"
        )
        task2_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task2_sim_score = evaluate_sim(orig_file_contents, task2_result, "task2")
        task2_evades = task2_sim_score is not None and task2_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 2 RESULTS:")
        print(f"Functional Correctness: {'✅ YES' if task2_functionally_correct else '❌ NO' if task2_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task2_sim_score:.4f}" if task2_sim_score else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task2_evades else '❌ NO'}")
        print(f"Model Used: {TASK2_MODEL}")
        print(f"Target Gates: {TASK2_TARGET_GATES}")

        # Overall success requires BOTH functional correctness AND SIM evasion
        overall_success = task2_functionally_correct and task2_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 2 failed: {e}")
        task2_result = None
        task2_sim_score = None
        task2_evades = False
        task2_functionally_correct = None
else:
    print("⚠️ Skipping Task 2 execution (no API key)")


Boolean Format Conversion:
Boolean equations: 212 lines
Remaining structural lines: 13 lines

First 300 chars of Boolean format:
KeyWire_0[0] = NOT(N1)
N118 = XNOR(keyIn_0_0, KeyWire_0[0])
N119 = NOT(N4)
KeyWire_0[1] = NOT(N11)
N122 = XNOR(keyIn_0_1, KeyWire_0[1])
N123 = NOT(N17)
KeyWire_0[2] = NOT(N24)
KeyNOTWire_0[0] = XNOR(keyIn_0_2, KeyWire_0[2])
N126 = NOT(KeyNOTWire_0[0])
N127 = NOT(N30)
KeyWire_0[3] = NOT(N37)
N130 = X...

Task 2 Configuration:
Target gates: NOR
Model: gpt-4
Prompt length: 6942 characters

EXECUTING TASK 2...
✅ LLM Response received (5352 chars)
First 300 chars of Boolean response:
KeyWire_0[0] = NOR(N1, N1)
N118 = NOR(NOR(keyIn_0_0, KeyWire_0[0]), NOR(NOR(keyIn_0_0, KeyWire_0[0]), KeyWire_0[0]))
N119 = NOR(N4, N4)
KeyWire_0[1] = NOR(N11, N11)
N122 = NOR(NOR(keyIn_0_1, KeyWire_0[1]), NOR(NOR(keyIn_0_1, KeyWire_0[1]), KeyWire_0[1]))
N123 = NOR(N17, N17)
KeyWire_0[2] = NOR(N24, ...

🔄 Converting back to Verilog format...
✅ Converted back to Verilog (6794 chars)

🔍

Model used:
GPT-3.5-turbo-16k and GPT-4

SIM Score:
0.2200 for both models

Success/Failure:
Failure overall for both models.

Key insights:
Both models evaded the SIM detector, but functional correctness remained unknown because the converted Verilog had syntax errors. GPT-4 performed better than GPT-3.5 because it generated a much fuller Boolean rewrite, while GPT-3.5’s response was clearly incomplete.



---

1. Success Rate

No. Neither GPT-3.5 nor GPT-4 successfully completed Task 2 overall. Both had Overall Success = NO. GPT-3.5 gave a very short incomplete response, while GPT-4 produced a fuller rewrite, but both failed the functionality check due to a syntax error.

2. SIM Evasion

The SIM score was 0.2200 for both models. Since this is below 0.3, both outputs evaded detection.

3. Functional Correctness

Functional correctness was unknown for both models. Yosys reported syntax error, unexpected TOK_INPUT, expecting ';', which means the converted Verilog was not valid.

4. Prompt Engineering

Compared with Task 1, the Task 2 prompt was shorter and more structured. It asked for Boolean equations instead of full Verilog, which should be easier for the model, but the conversion back to Verilog still caused syntax problems.

5. Model Comparison

GPT-4 performed better than GPT-3.5 in terms of output completeness. GPT-3.5 returned only a very short response, while GPT-4 generated a much fuller Boolean rewrite. However, both had Overall Success = NO because both failed functional verification.

In [13]:
#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Gates skipped by design: trivial (NOT/BUF) need no rephrasing
SKIP_GATE_TYPES = ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")

# TODO: Students configure task 3
TASK3_TARGET_GATES = ["OR", "NOT"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "gpt-3.5-turbo-16k"

# TODO: Students should improve this prompt (from original GPT_communication_script.py)
def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
        if target_gates[0] == "NOR":
            extra_rule = "Implement inversion as NOR(x, x)."
        elif target_gates[0] == "NAND":
            extra_rule = "Implement inversion as NAND(x, x)."
        else:
            extra_rule = ""
    else:
        gate_rule = f"Use only these Boolean operators: {', '.join(target_gates)}."
        extra_rule = ""

    return f"""
Rewrite the following Boolean gate implementation into an equivalent implementation.

Requirements:
1. {gate_rule}
2. {extra_rule}
3. Preserve the exact same functionality.
4. Use the same external variable names: Y for output and A1, A2, A3, ... for inputs.
5. You may introduce intermediate variables if needed.
6. Output ONLY Boolean equations, one per line.
7. Do NOT include explanations.
8. Do NOT include markdown code fences.
9. Each line must follow this exact format:
   variable = OPERATOR(input1, input2)
   or
   variable = OPERATOR(input1)
10. Do not use any operators outside the allowed target gate set.

Template:
{template}
"""

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = openai.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task3_sim_score:.4f}" if task3_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 execution (no API key)")

Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial, no rephrasing needed): ['not_1']

Task 3 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-3.5-turbo-16k

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Mapping: my_N84 = OR(A1, A2)
Y = NOT(my_N84)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Mapping: my_N86 = NOT(A2)
my_N87 = AND(A1, my_N86)
my_N88 = NOT(A1)
my_N89 = AND(my_N88, A2)
Y = OR(my_N87, my_N89)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Mapping: my_N91 = OR(A1, A2)
Y = NOT(my_N91)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Mapping: my_N93 = AND(A1, A2)
my_N94 = NOT(A1)
my_N95 = NOT(A2)
my_N96 = AND(my_N94, my_N95)
my_N97 = OR(my_N93, my_N96)
Y = NOT(my_N97)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A

In [14]:
#@title Task 3: Divide & Conquer (No Feedback) Implementation

# Characterize the circuit to get all gate types and query templates
# (This is the "divide" step: one LLM query per gate type)
gate_counts = characterize_generic_netlist(orig_file_contents)
query_templates = create_LLM_circuit_query_template(gate_counts)

# Gates skipped by design: trivial (NOT/BUF) need no rephrasing
SKIP_GATE_TYPES = ['not_1', 'buf_1']
non_trivial_gates = [g for g in gate_counts if g not in SKIP_GATE_TYPES]
skipped_gates    = [g for g in gate_counts if g in SKIP_GATE_TYPES]

print("Gate types found:", list(gate_counts.keys()))
print(f"Non-trivial gate types to map: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial, no rephrasing needed): {skipped_gates}")

# TODO: Students configure task 3
TASK3_TARGET_GATES = ["OR", "NOT"]  # Try: ["NOR"], ["NAND"], ["OR", "NOT"]
TASK3_MODEL = "gpt-4"

# TODO: Students should improve this prompt (from original GPT_communication_script.py)
def create_task3_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
        if target_gates[0] == "NOR":
            extra_rule = "Implement inversion as NOR(x, x)."
        elif target_gates[0] == "NAND":
            extra_rule = "Implement inversion as NAND(x, x)."
        else:
            extra_rule = ""
    else:
        gate_rule = f"Use only these Boolean operators: {', '.join(target_gates)}."
        extra_rule = ""

    return f"""
Rewrite the following Boolean gate implementation into an equivalent implementation.

Requirements:
1. {gate_rule}
2. {extra_rule}
3. Preserve the exact same functionality.
4. Use the same external variable names: Y for output and A1, A2, A3, ... for inputs.
5. You may introduce intermediate variables if needed.
6. Output ONLY Boolean equations, one per line.
7. Do NOT include explanations.
8. Do NOT include markdown code fences.
9. Each line must follow this exact format:
   variable = OPERATOR(input1, input2)
   or
   variable = OPERATOR(input1)
10. Do not use any operators outside the allowed target gate set.

Template:
{template}
"""

print(f"\nTask 3 Configuration:")
print(f"Target gates: {TASK3_TARGET_GATES}")
print(f"Model: {TASK3_MODEL}")

# Initialize results
task3_circuit_mapping = {}
task3_final_circuit = None
task3_sim_score = None
task3_evades = False
task3_functionally_correct = None

# Task 3 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 3...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK3_TARGET_GATES)

        # Initialize all gate types with empty mappings (unmapped gates stay as-is)
        for gate_type in gate_counts:
            task3_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, ask the LLM for a single-shot rephrasing
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            prompt = create_task3_prompt(template, TASK3_TARGET_GATES)

            response = openai.chat.completions.create(
                model=TASK3_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
                max_tokens=500
            )

            llm_response = response.choices[0].message.content
            LLM_circuit = get_circuit_from_response(llm_response)

            if "Incorrect format" in LLM_circuit:
                print(f"  ⚠️ Could not parse LLM response — keeping original gate")
                continue

            print(f"  ✅ Mapping: {LLM_circuit.strip()}")
            task3_circuit_mapping[gate_type][target_key] = [LLM_circuit]

        mapped_count = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")

        # Apply all mappings to the full circuit ("conquer" step)
        print("\n🔄 Applying gate mappings to full circuit...")
        task3_final_circuit = get_mapped_circuit(orig_file_contents, task3_circuit_mapping, target_key)

        # Check functional equivalence
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task3_final_circuit, "task3"
        )
        task3_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task3_sim_score = evaluate_sim(orig_file_contents, task3_final_circuit, "task3")
        task3_evades = task3_sim_score is not None and task3_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 3 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Functional Correctness: {'✅ YES' if task3_functionally_correct else '❌ NO' if task3_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task3_sim_score:.4f}" if task3_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task3_evades else '❌ NO'}")
        print(f"Model Used: {TASK3_MODEL}")
        print(f"Target Gates: {TASK3_TARGET_GATES}")
        overall_success = task3_functionally_correct and task3_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

    except Exception as e:
        print(f"❌ Task 3 failed: {e}")
        task3_circuit_mapping = {}
        task3_final_circuit = None
        task3_sim_score = None
        task3_evades = False
        task3_functionally_correct = None
else:
    print("⚠️ Skipping Task 3 execution (no API key)")

Gate types found: ['nand_2', 'not_1', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Non-trivial gate types to map: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial, no rephrasing needed): ['not_1']

Task 3 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-4

EXECUTING TASK 3...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2)
Y = OR(N1, N2)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  ✅ Mapping: B1 = NOT(A1)
B2 = NOT(A2)
C1 = OR(A1, B2)
C2 = OR(B1, A2)
Y = OR(C1, C2)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  ✅ Mapping: B1 = NOT(A1)
B2 = NOT(A2)
Y = OR(B1, B2)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2)
O1 = OR(A1, N2)
O2 = OR(A2, N1)
my_N146 = NOT(O1)
my_N147 = NOT(O2)
Y = OR(my_N146, my_N147)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2

Model used:
GPT-3.5-turbo-16k and GPT-4

SIM Score:
0.2500 for both models

Success/Failure:
Failure overall for both models.

Key insights:
Both models successfully mapped all non-trivial gate types and evaded the SIM detector, but both failed functional equivalence. This suggests that divide-and-conquer improved structure compared with rewriting the whole circuit at once, but incorrect local gate mappings still caused the final circuit to be wrong.


---

1. Success Rate

No. Neither GPT-3.5 nor GPT-4 successfully completed Task 3 overall. Both mapped all 8 non-trivial gate types, but the final rewritten circuit was not equivalent to the original one.

2. SIM Evasion

The SIM score was 0.2500 for both models. Since this is below 0.3, both outputs evaded detection.

3. Functional Correctness

Functional correctness was NO for both models. Yosys SAT found a counterexample, which means the rewritten circuit was not functionally equivalent. This happened because some learned gate mappings were logically incorrect, even though they looked syntactically valid.

4. Prompt Engineering

Compared with Task 2, Task 3 uses a divide-and-conquer strategy: instead of rewriting the whole circuit at once, it rewrites one gate template at a time and then applies those mappings to the full circuit. This makes the task more structured, but if even one gate mapping is wrong, the final circuit will fail.

5. Model Comparison

GPT-3.5 and GPT-4 performed very similarly here. Both mapped all gate types, both got the same SIM score, and both failed functional equivalence. GPT-4’s mappings looked slightly cleaner, but neither model achieved overall success. The Task 3 notebook structure confirms this divide-and-conquer flow and the per-gate mapping setup.

In [16]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3
# (gate_counts, query_templates, SKIP_GATE_TYPES,
#  non_trivial_gates, skipped_gates are already set)
print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# TODO: Students configure task 4
TASK4_TARGET_GATES = ["OR", "NOT"]
TASK4_MODEL = "gpt-4"
TASK4_MAX_TRIALS = 5

def create_task4_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
        if target_gates[0] == "NOR":
            extra_rule = "Implement inversion as NOR(x, x)."
        elif target_gates[0] == "NAND":
            extra_rule = "Implement inversion as NAND(x, x)."
        else:
            extra_rule = ""
    else:
        gate_rule = f"Use only these Boolean operators: {', '.join(target_gates)}."
        extra_rule = ""

    return f"""
Rewrite the following gate template into an equivalent Boolean implementation.

Requirements:
1. {gate_rule}
2. {extra_rule}
3. Preserve the exact same functionality.
4. Keep the external variable names unchanged: Y for output, A1/A2/A3... for inputs.
5. You may introduce intermediate variables if needed.
6. Output ONLY Boolean equations, one per line.
7. Do NOT include explanations.
8. Do NOT include markdown code fences.
9. Each line must follow one of these exact formats:
   x = OP(a, b)
   x = OP(a)
10. Do not use any operators outside the allowed target gate set.

Template:
{template}
"""

def create_task4_format_feedback(template, target_gates):
    return f"""
Your previous answer was not in the required format.

Try again and follow these rules exactly:
1. Output ONLY Boolean equations, one per line.
2. Do NOT include explanations.
3. Do NOT include markdown code fences.
4. Use only these target gates: {', '.join(target_gates)}.
5. Keep Y as the output and A1/A2/A3... as the inputs.
6. Each line must follow one of these exact formats:
   x = OP(a, b)
   x = OP(a)

Original template:
{template}
"""

def create_task4_functional_feedback(template, target_gates):
    return f"""
Your previous answer did not preserve the same functionality.

Please try again.

Rules:
1. Preserve exact functionality.
2. Use only these target gates: {', '.join(target_gates)}.
3. Keep Y as the output and A1/A2/A3... as the inputs.
4. Output ONLY Boolean equations, one per line.
5. Do NOT include explanations or markdown code fences.
6. Each line must follow:
   x = OP(a, b)
   x = OP(a)

Original template:
{template}
"""

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = openai.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.7,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task4_sim_score:.4f}" if task4_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")

    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 execution (no API key)")


Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-4
Max trials per gate type: 5

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2)
Y = OR(N1, N2)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: X1 = NOT(A1)
X2 = NOT(A2)
Y1 = OR(A1, X2)
Y2 = OR(A2, X1)
Y = OR(Y1, Y2)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: my_N201 = OR(A1, A2)
Y = NOT(my_N201)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2)
O1 = OR(A1, A2)
O2 = OR(N1, N2)
Y = OR(O1, O2)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  Trial 1/5
  ✅ Mapping: N1 = NOT(A1)
N2 = NOT(A2)
N3 = NOT(A3)
N4 = NOT(A4)
Y = OR(N1, N2, N3, N4)

--- Gate type: and_9  (template: Y = AND(A1

In [17]:
#@title Task 4: Divide & Conquer WITH Iterative Feedback Implementation

# Reuse gate characterization from Task 3
# (gate_counts, query_templates, SKIP_GATE_TYPES,
#  non_trivial_gates, skipped_gates are already set)
print(f"Gate types to process: {non_trivial_gates}")
if skipped_gates:
    print(f"Skipped (trivial): {skipped_gates}")

# TODO: Students configure task 4
TASK4_TARGET_GATES = ["OR", "NOT"]
TASK4_MODEL = "gpt-3.5-turbo-16k"
TASK4_MAX_TRIALS = 5

def create_task4_prompt(template, target_gates):
    if len(target_gates) == 1:
        gate_rule = f"Use only the {target_gates[0]} Boolean operator."
        if target_gates[0] == "NOR":
            extra_rule = "Implement inversion as NOR(x, x)."
        elif target_gates[0] == "NAND":
            extra_rule = "Implement inversion as NAND(x, x)."
        else:
            extra_rule = ""
    else:
        gate_rule = f"Use only these Boolean operators: {', '.join(target_gates)}."
        extra_rule = ""

    return f"""
Rewrite the following gate template into an equivalent Boolean implementation.

Requirements:
1. {gate_rule}
2. {extra_rule}
3. Preserve the exact same functionality.
4. Keep the external variable names unchanged: Y for output, A1/A2/A3... for inputs.
5. You may introduce intermediate variables if needed.
6. Output ONLY Boolean equations, one per line.
7. Do NOT include explanations.
8. Do NOT include markdown code fences.
9. Each line must follow one of these exact formats:
   x = OP(a, b)
   x = OP(a)
10. Do not use any operators outside the allowed target gate set.

Template:
{template}
"""

def create_task4_format_feedback(template, target_gates):
    return f"""
Your previous answer was not in the required format.

Try again and follow these rules exactly:
1. Output ONLY Boolean equations, one per line.
2. Do NOT include explanations.
3. Do NOT include markdown code fences.
4. Use only these target gates: {', '.join(target_gates)}.
5. Keep Y as the output and A1/A2/A3... as the inputs.
6. Each line must follow one of these exact formats:
   x = OP(a, b)
   x = OP(a)

Original template:
{template}
"""

def create_task4_functional_feedback(template, target_gates):
    return f"""
Your previous answer did not preserve the same functionality.

Please try again.

Rules:
1. Preserve exact functionality.
2. Use only these target gates: {', '.join(target_gates)}.
3. Keep Y as the output and A1/A2/A3... as the inputs.
4. Output ONLY Boolean equations, one per line.
5. Do NOT include explanations or markdown code fences.
6. Each line must follow:
   x = OP(a, b)
   x = OP(a)

Original template:
{template}
"""

print(f"\nTask 4 Configuration:")
print(f"Target gates: {TASK4_TARGET_GATES}")
print(f"Model: {TASK4_MODEL}")
print(f"Max trials per gate type: {TASK4_MAX_TRIALS}")

# Initialize results
task4_circuit_mapping = {}
task4_final_circuit = None
task4_sim_score = None
task4_evades = False
task4_functionally_correct = None
task4_gate_trial_counts = {}

# Task 4 Execution
if OPENAI_API_KEY:
    print("\n" + "=" * 50)
    print("EXECUTING TASK 4...")
    print("=" * 50)

    try:
        target_key = "_".join(TASK4_TARGET_GATES)

        # Initialize all gate types with empty mappings
        for gate_type in gate_counts:
            task4_circuit_mapping[gate_type] = {}

        # For each non-trivial gate type, query LLM with iterative feedback on failure
        for gate_type, template in query_templates.items():
            if gate_type in SKIP_GATE_TYPES:
                continue

            print(f"\n--- Gate type: {gate_type}  (template: {template}) ---")
            messages = [{"role": "user", "content": create_task4_prompt(template, TASK4_TARGET_GATES)}]
            num_trials = 0
            success = False

            while num_trials < TASK4_MAX_TRIALS:
                num_trials += 1
                print(f"  Trial {num_trials}/{TASK4_MAX_TRIALS}")

                # Use full conversation history for context (feedback loop)
                response = openai.chat.completions.create(
                    model=TASK4_MODEL,
                    messages=messages,
                    temperature=0.7,
                    max_tokens=500
                )
                llm_response = response.choices[0].message.content
                messages.append({"role": "assistant", "content": llm_response})

                LLM_circuit = get_circuit_from_response(llm_response)

                if "Incorrect format" in LLM_circuit:
                    print(f"  ⚠️ Format error — sending feedback")
                    feedback = create_task4_format_feedback(template, TASK4_TARGET_GATES)
                    messages.append({"role": "user", "content": feedback})
                    continue

                print(f"  ✅ Mapping: {LLM_circuit.strip()}")
                task4_circuit_mapping[gate_type][target_key] = [LLM_circuit]
                success = True
                break

            task4_gate_trial_counts[gate_type] = num_trials
            if not success:
                print(f"  ❌ No valid mapping after {TASK4_MAX_TRIALS} trials — keeping original gate")

        mapped_count = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
        avg_trials = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
        print(f"\n✅ Got mappings for {mapped_count}/{len(non_trivial_gates)} non-trivial gate types")
        if skipped_gates:
            print(f"   (Skipped by design: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")

        # Apply all mappings to the full circuit
        print("\n🔄 Applying gate mappings to full circuit...")
        task4_final_circuit = get_mapped_circuit(orig_file_contents, task4_circuit_mapping, target_key)

        # Check functional equivalence on the full mapped circuit
        print("\n🔍 Checking functional equivalence...")
        equiv_result, equiv_details = check_functional_equivalence(
            orig_file_contents, task4_final_circuit, "task4"
        )
        task4_functionally_correct = equiv_result
        print(f"Functional check: {equiv_details}")

        # Evaluate with SIM
        print("\n🔍 Evaluating with SIM detector...")
        task4_sim_score = evaluate_sim(orig_file_contents, task4_final_circuit, "task4")
        task4_evades = task4_sim_score is not None and task4_sim_score < SIM_THRESHOLD

        print(f"\n📊 TASK 4 RESULTS:")
        print(f"Gate types mapped: {mapped_count}/{len(non_trivial_gates)} non-trivial  (skipped: {skipped_gates})")
        print(f"Average trials per gate type: {avg_trials:.1f}")
        print(f"Functional Correctness: {'✅ YES' if task4_functionally_correct else '❌ NO' if task4_functionally_correct is False else '⚠️ UNKNOWN'}")
        print(f"SIM Score: {task4_sim_score:.4f}" if task4_sim_score is not None else "SIM Score: Error")
        print(f"Evades Detection: {'✅ YES' if task4_evades else '❌ NO'}")
        print(f"Model Used: {TASK4_MODEL}")
        print(f"Target Gates: {TASK4_TARGET_GATES}")
        overall_success = task4_functionally_correct and task4_evades
        print(f"Overall Success: {'✅ YES' if overall_success else '❌ NO'}")

        # Show per-gate-type trial counts
        print("\nPer-gate-type trial counts:")
        for gt in gate_counts:
            if gt in SKIP_GATE_TYPES:
                print(f"  {gt}: — (trivial gate, skipped by design)")
            elif gt in task4_gate_trial_counts:
                trials = task4_gate_trial_counts[gt]
                mapped = '✅' if task4_circuit_mapping.get(gt) else '❌'
                print(f"  {gt}: {trials} trial(s) {mapped}")

    except Exception as e:
        print(f"❌ Task 4 failed: {e}")
        task4_circuit_mapping = {}
        task4_final_circuit = None
        task4_sim_score = None
        task4_evades = False
        task4_functionally_correct = None
        task4_gate_trial_counts = {}
else:
    print("⚠️ Skipping Task 4 execution (no API key)")


Gate types to process: ['nand_2', 'xor_2', 'nor_2', 'xnor_2', 'nand_4', 'and_9', 'and_8', 'nand_3']
Skipped (trivial): ['not_1']

Task 4 Configuration:
Target gates: ['OR', 'NOT']
Model: gpt-3.5-turbo-16k
Max trials per gate type: 5

EXECUTING TASK 4...

--- Gate type: nand_2  (template: Y = NAND(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: my_N238 = OR(A1, A2)
Y = NOT(my_N238)

--- Gate type: xor_2  (template: Y = XOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: my_N240 = NOT(A2)
my_N241 = AND(A1, my_N240)
my_N242 = NOT(A1)
my_N243 = AND(my_N242, A2)
Y = OR(my_N241, my_N243)

--- Gate type: nor_2  (template: Y = NOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: my_N245 = OR(A1, A2)
Y = NOT(my_N245)

--- Gate type: xnor_2  (template: Y = XNOR(A1, A2)) ---
  Trial 1/5
  ✅ Mapping: my_N247 = AND(A1, A2)
my_N248 = NOT(A1)
my_N249 = NOT(A2)
my_N250 = AND(my_N248, my_N249)
my_N251 = OR(my_N247, my_N250)
Y = NOT(my_N251)

--- Gate type: nand_4  (template: Y = NAND(A1, A2, A3, A4)) ---
  Trial 1/5
  ✅ Mapping: tem

Model used:
GPT-3.5-turbo-16k and GPT-4

SIM Score:
0.2500 for both models

Success/Failure:
Failure overall for both models.

Key insights:
Both models mapped all non-trivial gate types and evaded the SIM detector, but both failed functional equivalence. Although Task 4 adds iterative feedback, in these runs every mapping was accepted on the first trial, so the feedback loop did not meaningfully improve results.


---

1. Success Rate

No. Neither GPT-3.5 nor GPT-4 achieved overall success in Task 4. Both models found mappings for all 8 non-trivial gate types using the target gates ['OR', 'NOT'], but the final rewritten circuit was still not functionally equivalent to the original. The Task 4 flow uses iterative retries per gate type, but in these runs every gate was accepted on the first trial, so the feedback loop did not actually correct any wrong mappings.

2. SIM Evasion

The SIM score was 0.2500 for both models. Since this is below 0.3, both outputs evaded detection.

3. Functional Correctness

Functional correctness was NO for both models. Yosys SAT found a counterexample, which means the final mapped circuit was not equivalent to the original. The likely reason is that some local gate transformations were logically incorrect, even though they looked valid in equation format.

4. Prompt Engineering

Compared with Task 3, Task 4 adds an iterative feedback loop, so in theory it should improve bad mappings by retrying. In these runs, however, every gate mapping was accepted in one trial, so Task 4 behaved almost the same as Task 3 and did not gain much from feedback.

5. Model Comparison

GPT-3.5 and GPT-4 performed very similarly here. Both mapped all gate types, both got the same SIM score, and both failed functional equivalence. GPT-4’s equations looked a little cleaner, but neither model performed better overall.

In [18]:
#@title Task 5: Reference Baseline + Strategy Comparison

import pandas as pd
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# PART A: Reference Baseline — Original Research Cached Mappings
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("PART A: REFERENCE BASELINE")
print("=" * 60)

# TODO: Students can switch LLM to compare baselines
REFERENCE_LLM = 'GPT3dot5'   # Try: 'GPT3dot5', 'GPT4', 'Claude', 'Gemini', 'llama3'
REFERENCE_STRATEGY = 'random'  # 'random' picks best available per gate type

mapping_path = os.path.join(DATA_DIR, 'src', f'cached_circuit_mapping_{REFERENCE_LLM}.pkl')
with open(mapping_path, 'rb') as f:
    reference_mapping = pickle.load(f)

print(f"Loaded cached mapping: {REFERENCE_LLM}")
print(f"\nCoverage for this circuit's gate types:")
for gt in gate_counts:
    strategies = reference_mapping.get(gt, {})
    if strategies:
        print(f"  ✅ {gt}: {list(strategies.keys())}")
    else:
        print(f"  ⬜ {gt}: no cached mapping (gate kept as-is)")

print(f"\n🔄 Applying {REFERENCE_LLM} cached mappings (strategy: {REFERENCE_STRATEGY})...")
ref_circuit = get_mapped_circuit(orig_file_contents, reference_mapping, REFERENCE_STRATEGY)

print("\n🔍 Checking functional equivalence...")
ref_equiv, ref_details = check_functional_equivalence(orig_file_contents, ref_circuit, 'reference')
print(f"Functional check: {ref_details}")

print("\n🔍 Evaluating with SIM detector...")
ref_sim_score = evaluate_sim(orig_file_contents, ref_circuit, 'reference')
ref_evades = ref_sim_score is not None and ref_sim_score < SIM_THRESHOLD

print(f"\n📊 REFERENCE BASELINE RESULTS ({REFERENCE_LLM}):")
print(f"Functional Correctness: {'✅ YES' if ref_equiv else '❌ NO' if ref_equiv is False else '⚠️ UNKNOWN'}")
print(f"SIM Score: {ref_sim_score:.4f}" if ref_sim_score is not None else "SIM Score: Error")
print(f"Evades Detection: {'✅ YES' if ref_evades else '❌ NO'}")

# ─────────────────────────────────────────────────────────────
# PART B: Strategy Comparison
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B: COMPREHENSIVE STRATEGY COMPARISON")
print("=" * 60)

all_results = []

# Task 1 Results
if 'task1_result' in locals() and task1_sim_score is not None:
    all_results.append({
        'Task': 'Task 1: Direct Verilog',
        'Method': 'Direct LLM rewriting',
        'SIM_Score': task1_sim_score,
        'Functional': task1_functionally_correct,
        'Overall_Success': bool(task1_functionally_correct and task1_evades),
        'Model': TASK1_MODEL,
        'Notes': 'Single-shot, full circuit'
    })

# Task 2 Results
if 'task2_result' in locals() and task2_sim_score is not None:
    all_results.append({
        'Task': 'Task 2: Boolean Format',
        'Method': 'Boolean intermediate format',
        'SIM_Score': task2_sim_score,
        'Functional': task2_functionally_correct,
        'Overall_Success': bool(task2_functionally_correct and task2_evades),
        'Model': TASK2_MODEL,
        'Notes': 'Via Boolean equations'
    })

# Task 3 Results
if 'task3_final_circuit' in locals() and task3_final_circuit is not None:
    mapped_count3 = sum(1 for g in non_trivial_gates if task3_circuit_mapping.get(g))
    all_results.append({
        'Task': 'Task 3: D&C No Feedback',
        'Method': 'Gate-type mapping, single-shot',
        'SIM_Score': task3_sim_score,
        'Functional': task3_functionally_correct,
        'Overall_Success': bool(task3_functionally_correct and task3_evades),
        'Model': TASK3_MODEL,
        'Notes': f'{mapped_count3}/{len(non_trivial_gates)} gate types mapped'
    })

# Task 4 Results
if 'task4_final_circuit' in locals() and task4_final_circuit is not None:
    mapped_count4 = sum(1 for g in non_trivial_gates if task4_circuit_mapping.get(g))
    avg_trials4 = sum(task4_gate_trial_counts.values()) / len(task4_gate_trial_counts) if task4_gate_trial_counts else 0
    all_results.append({
        'Task': 'Task 4: D&C With Feedback',
        'Method': 'Gate-type mapping, iterative feedback',
        'SIM_Score': task4_sim_score,
        'Functional': task4_functionally_correct,
        'Overall_Success': bool(task4_functionally_correct and task4_evades),
        'Model': TASK4_MODEL,
        'Notes': f'{mapped_count4}/{len(non_trivial_gates)} mapped, {avg_trials4:.1f} avg trials'
    })

# Reference Baseline Results
covered_gates = sum(1 for gt in gate_counts if reference_mapping.get(gt))
all_results.append({
    'Task': f'Reference: {REFERENCE_LLM} Cache',
    'Method': 'Pre-validated research mappings',
    'SIM_Score': ref_sim_score,
    'Functional': ref_equiv,
    'Overall_Success': bool(ref_equiv and ref_evades),
    'Model': REFERENCE_LLM,
    'Notes': f'{covered_gates}/{len(gate_counts)} gate types in cache'
})

# ── Print comparison table ──────────────────────────────────
print("\n📋 SUMMARY TABLE:")
print("-" * 60)
for row in all_results:
    print(f"\n{row['Task']}:")
    print(f"  Method: {row['Method']}")
    print(f"  Model:  {row['Model']}")
    print(f"  SIM Score: {row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "  SIM Score: N/A")
    func = row['Functional']
    print(f"  Functional: {'✅ YES' if func else ('❌ NO' if func is False else '⚠️ UNKNOWN')}")
    print(f"  Overall Success: {'✅ YES' if row['Overall_Success'] else '❌ NO'}")
    print(f"  Notes: {row['Notes']}")

# ── Ranking ─────────────────────────────────────────────────
print(f"\n🎯 STRATEGY EFFECTIVENESS RANKING:")
print("-" * 40)

sorted_results = sorted(
    all_results,
    key=lambda r: (int(r['Overall_Success']), -(r['SIM_Score'] if r['SIM_Score'] is not None else 1.0))
)[::-1]

for rank, row in enumerate(sorted_results, 1):
    icon = '🏆' if rank == 1 else '🥈' if rank == 2 else '🥉' if rank == 3 else '📍'
    sim_str = f"SIM={row['SIM_Score']:.4f}" if row['SIM_Score'] is not None else "SIM=N/A"
    success_str = '✅' if row['Overall_Success'] else '❌'
    print(f"{rank}. {icon} {row['Task']:35s}  {success_str}  {sim_str}")

# ── Key insights ─────────────────────────────────────────────
print(f"\n📈 KEY INSIGHTS:")
valid_sim = [r for r in all_results if r['SIM_Score'] is not None]
if valid_sim:
    best = min(valid_sim, key=lambda r: r['SIM_Score'])
    print(f"• Lowest SIM score: {best['SIM_Score']:.4f}  ({best['Task']})")
    if ref_sim_score is not None:
        for row in all_results:
            if row['Task'].startswith('Task') and row['SIM_Score'] is not None:
                diff = row['SIM_Score'] - ref_sim_score
                indicator = '🔺 worse' if diff > 0.02 else ('🔻 better' if diff < -0.02 else '≈ similar')
                print(f"• {row['Task']:35s} vs reference: {diff:+.4f} ({indicator})")

model_results = defaultdict(list)
for r in all_results:
    model_results[r['Model']].append(r['Overall_Success'])
if len(model_results) > 1:
    print(f"\n🤖 MODEL COMPARISON:")
    for model, successes in model_results.items():
        rate = sum(successes) / len(successes)
        print(f"• {model}: {rate:.0%} success rate")


PART A: REFERENCE BASELINE
Loaded cached mapping: GPT3dot5

Coverage for this circuit's gate types:
  ✅ nand_2: ['AND_NOT', 'OR_NOT']
  ⬜ not_1: no cached mapping (gate kept as-is)
  ✅ xor_2: ['NAND']
  ✅ nor_2: ['AND_NOT']
  ✅ xnor_2: ['NOR']
  ✅ nand_4: ['NOR', 'AND_NOT', 'OR_NOT']
  ⬜ and_9: no cached mapping (gate kept as-is)
  ⬜ and_8: no cached mapping (gate kept as-is)
  ✅ nand_3: ['AND_NOT', 'OR_NOT']

🔄 Applying GPT3dot5 cached mappings (strategy: random)...

🔍 Checking functional equivalence...
🔍 Running Yosys equivalence check for 'reference'...
Functional check: Circuits formally verified equivalent (Yosys miter+SAT)

🔍 Evaluating with SIM detector...

📊 REFERENCE BASELINE RESULTS (GPT3dot5):
Functional Correctness: ✅ YES
SIM Score: 0.2500
Evades Detection: ✅ YES

PART B: COMPREHENSIVE STRATEGY COMPARISON

📋 SUMMARY TABLE:
------------------------------------------------------------

Task 1: Direct Verilog:
  Method: Direct LLM rewriting
  Model:  gpt-4
  SIM Score: 0.2200


**Final Analysis**

Across Tasks 1–4, none of my strategies achieved full overall success. The main problems were syntax errors, incomplete outputs, and incorrect gate mappings that caused the rewritten circuits to fail equivalence checking. However, several methods still achieved low SIM scores, so the LLMs were often able to change the circuit structure enough to evade similarity-based detection.

For strategy effectiveness, Tasks 3 and 4 were more structured than Tasks 1 and 2 because they rewrote the circuit by gate type instead of rewriting the whole design at once. The most common errors were invalid Verilog, wrong XOR/XNOR rewrites, and mistakes on large multi-input gates.

GPT-4 did not significantly outperform GPT-3.5 in my experiments. GPT-4 often gave cleaner outputs, but the final results were usually similar because both models still failed functional correctness.

For target gates, OR + NOT worked better than pure NOR because the rewrites were simpler and easier for the model to produce correctly. More complex gate sets like NOR-only required more indirect constructions, which increased errors.

Task 4 did not improve much over Task 3 in my runs. Although it added feedback, most mappings were accepted on the first trial, so the feedback loop did not really fix the main functional errors.

In practice, I would recommend divide-and-conquer rewriting over direct full-circuit rewriting, because it is more structured and easier to analyze. The main trade-off is that even one wrong local mapping can break the whole circuit, so formal equivalence checking is still necessary.


---



**Best strategy overall:**
Task 3 / Task 4


---



Key insights:
SIM evasion was easier than preserving exact functionality.


---



Surprising findings:
GPT-4 was not much better than GPT-3.5 in final success.


---



Recommendations:
Use divide-and-conquer methods with simpler target gates like OR + NOT, and always verify with formal checking.